# Proyecto final: Sistema de Inferencia Difusa para Control de Lavavajillas

Autor: Esteban Suárez Calvo

In [27]:
import numpy as np

## Especificación del problema

# Proyecto Final: Sistema de Inferencia Difusa para Control de Lavavajillas

El objetivo de este proyecto es desarrollar un modelo de inferencia difusa que determine el modo de lavado óptimo de un lavavajillas en base a la carga de platos y al nivel de grasa.

## Variables de entrada

Las variables de entrada son las siguientes:

1. **Carga de platos (0-100):** representa el porcentaje de ocupación del lavavajillas. Las etiquetas son las siguientes:
   1. *Poca*
   2. *Media*
   3. *Mucha*
2. **Nivel de grasa (0-100):** representa la cantidad de grasa que tienen los platos. Las etiquetas son las siguientes:
   1. *Ligera*
   2. *Normal*
   3. *Intensa*

## Variable de salida

La salida del sistema es una única variable, el **modo de lavado**, que tema las siguientes etiquetas:

1. *Eco* (30 min)
2. *Normal* (60 min)
3. *Intensivo* (90 min)

## Base de reglas

El sistema agrupa 9 reglas en las siguientes combinaciones mediante operadores AND y OR:

- SI (Poca Y Ligera) O (Poca Y Normal) O (Media Y Ligera) ENTONCES **Modo Eco**
- SI (Poca Y Intensa) O (Media Y Normal) O (Mucha Y Ligera) ENTONCES **Modo Eco**
- SI (Media Y Intensa) O (Poca Y Normal) O (Media Y Ligera) ENTONCES **Modo Eco**

## Declaración del sistema clásico

Primero de todo, definimos las funciones de pertenencia base que utilizaremos más adelante.

In [28]:
def linzmf(x, a, b):
    if x <= a:
        return 1.0
    if x >= b:
        return 0.0
    return (b - x) / (b - a)


def linsmf(x, a, b):
    if x <= a:
        return 0.0
    if x >= b:
        return 1.0
    return (x - a) / (b - a)


def trimf(x, a, b, c):
    if x <= a or x >= c:
        return 0.0
    if a < x <= b:
        return (x - a) / (b - a)
    if b < x < c:
        return (c - x) / (c - b)
    return 0.0

Utilizando estas funciones de pertenencia definimos las variables difusas.

In [29]:
def fuzzy_carga(x):
    return {
        "poca": linzmf(x, 20, 50),
        "media": trimf(x, 20, 50, 80),
        "mucha": linsmf(x, 50, 80),
    }


def fuzzy_grasa(x):
    return {
        "ligera": linzmf(x, 20, 50),
        "normal": trimf(x, 20, 50, 80),
        "intensa": linsmf(x, 50, 80),
    }

Realizamos entonces la inferencia clásica tipo Sugeno. Destacar que para las reglas comobinadas utilizamos `min` para realizar las operaciones AND y `max` para realizar las operaciones OR.

In [ ]:
def inferencia_clasica_lavavajillas(val_carga, val_grasa):
    carga = fuzzy_carga(val_carga)
    grasa = fuzzy_grasa(val_grasa)

    # Valores numéricos de la salida (minutos)
    out_eco = 30
    out_normal = 60
    out_intensivo = 90

    # 1. Evaluamos las reglas (Pesos de activación)
    w_eco = max(
        min(carga["poca"], grasa["ligera"]),
        min(carga["poca"], grasa["normal"]),
        min(carga["media"], grasa["ligera"]),
    )

    w_normal = max(
        min(carga["poca"], grasa["intensa"]),
        min(carga["media"], grasa["normal"]),
        min(carga["mucha"], grasa["ligera"]),
    )

    w_intensivo = max(
        min(carga["media"], grasa["intensa"]),
        min(carga["mucha"], grasa["normal"]),
        min(carga["mucha"], grasa["intensa"]),
    )

    pesos = [w_eco, w_normal, w_intensivo]

    # 2. OBTENER LA CATEGORÍA (El peso más alto gana)
    etiquetas = ["Eco", "Normal", "Intensivo"]
    indice_ganador = np.argmax(pesos)
    categoria_final = etiquetas[indice_ganador]

    # 3. OBTENER LOS MINUTOS EXACTOS (Defuzzificación por media ponderada)
    denominador = sum(pesos)
    minutos_exactos = (
        (w_eco * out_eco) + (w_normal * out_normal) + (w_intensivo * out_intensivo)
    ) / denominador

    return categoria_final, minutos_exactos, pesos

Por último probamos el sistema:

In [35]:
carga_test = 65
grasa_test = 40

categoria, minutos, pesos = inferencia_clasica_lavavajillas(carga_test, grasa_test)

print(f"Para una carga del {carga_test}% y una grasa del {grasa_test}%:")
print(f"Categoría recomendada: Modo {categoria}")
print(f"Tiempo exacto calculado: {minutos:.1f} minutos")
print(f"Pesos [Eco, Normal, Intensivo]={pesos}")

Para una carga del 65% y una grasa del 40%:
Categoría recomendada: Modo Normal
Tiempo exacto calculado: 63.8 minutos
Pesos [Eco, Normal, Intensivo]=[0.3333333333333333, 0.5, 0.5]


## Implementación del sistema cuántico